# Wilson-loop potential fit plot

This notebook reads the precomputed Wilson-potential output and plots the stored single-exponential fit
\(W(R,T)=c_0(R)\exp[-V(R)T]\) together with the Wilson-loop data. The figure is saved as a PDF.

In [10]:
from pathlib import Path
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.optimize import curve_fit

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from finalized_analysis_helpers import _ensure_plotly_browser_path

POTENTIAL_DIR = Path("../data/gradient_flow_wtemp_analysis/beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58/potential_t_over_a2_0p5")
R_VALUE = 4
PDF_PATH_LINEAR = POTENTIAL_DIR / f"wilson_loop_fit_R{R_VALUE}_linear.pdf"
PDF_PATH_LOG = POTENTIAL_DIR / f"wilson_loop_fit_R{R_VALUE}_log.pdf"

# Set these to tuples like (xmin, xmax) / (ymin, ymax), or leave as None for automatic limits.
LINEAR_XLIM = (9.8, 18.5)
LINEAR_YLIM = (0.02, 0.15)

# Plot styling, matching effective_mass_plotting.ipynb PDF exports.
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600
POINT_COLOR = "#1f77b4"
FIT_COLOR = "#d62728"
EXCLUDED_COLOR = "#7f7f7f"
CAPSIZE = 3
MARKER_SIZE = 7
PDF_MARGIN = {"l": 48, "r": 12, "t": 12, "b": 46}

_ensure_plotly_browser_path()
pio.templates.default = "plotly_white"

wrt_path = POTENTIAL_DIR / "wrt_scan.json"
summary_path = POTENTIAL_DIR / "V_R_summary.json"

with wrt_path.open("r", encoding="utf-8") as handle:
    wrt_records = json.load(handle)
with summary_path.open("r", encoding="utf-8") as handle:
    v_summary = json.load(handle)

fit_info = v_summary["results"].get(str(R_VALUE))
if fit_info is None:
    available = ", ".join(sorted(v_summary.get("results", {}).keys(), key=int))
    raise ValueError(f"No finalized V(R) result for R={R_VALUE}. Available R values: {available}")

rows = sorted(
    [row for row in wrt_records if int(row["R"]) == int(R_VALUE)],
    key=lambda row: int(row["T"]),
)
if not rows:
    raise ValueError(f"No Wilson-loop data found for R={R_VALUE}")

T = np.asarray([int(row["T"]) for row in rows], dtype=float)
W = np.asarray([float(row["value"]) for row in rows], dtype=float)
W_bootstrap = [np.asarray(np.load(row["bootstrap_path"]), dtype=float) for row in rows]
W_err = np.asarray(
    [float(row["err"]) if row.get("err") is not None else np.nan for row in rows],
    dtype=float,
)
missing_err = ~np.isfinite(W_err) | (W_err < 0)
if np.any(missing_err):
    W_err[missing_err] = [
        float(np.std(samples[np.isfinite(samples)])) if np.any(np.isfinite(samples)) else np.nan
        for samples, missing in zip(W_bootstrap, missing_err)
        if missing
    ]

fit_T = np.asarray(fit_info.get("fit_T", []), dtype=int)
fit_mask = np.isin(T.astype(int), fit_T)

C = float(fit_info["fit_C"])
V = float(fit_info["value"])

t_line = np.linspace(float(np.min(T)), float(np.max(T)), 500)
w_fit = C * np.exp(-V * t_line)

def exponential_model(t, c_0, v):
    return c_0 * np.exp(-v * np.asarray(t, dtype=float))

fit_residuals = W[fit_mask] - C * np.exp(-V * T[fit_mask])
fit_sigmas = W_err[fit_mask]
chi_mask = np.isfinite(fit_residuals) & np.isfinite(fit_sigmas) & (fit_sigmas > 0)
chi2 = float(np.sum((fit_residuals[chi_mask] / fit_sigmas[chi_mask]) ** 2)) if np.any(chi_mask) else None
chi2_dof = chi2 / (int(np.sum(chi_mask)) - 2) if chi2 is not None and int(np.sum(chi_mask)) > 2 else None

bootstrap_arrays = [samples for samples, is_fit in zip(W_bootstrap, fit_mask) if is_fit]
n_bootstrap = min(array.size for array in bootstrap_arrays) if bootstrap_arrays else 0
bootstrap_c = np.full(n_bootstrap, np.nan, dtype=float)
bootstrap_v = np.full(n_bootstrap, np.nan, dtype=float)

for idx in range(n_bootstrap):
    w_sample = np.asarray([array[idx] for array in bootstrap_arrays], dtype=float)
    valid = np.isfinite(w_sample) & (w_sample > 0) & np.isfinite(fit_sigmas) & (fit_sigmas > 0)
    if np.count_nonzero(valid) < 3:
        continue
    try:
        popt, _ = curve_fit(
            exponential_model,
            T[fit_mask][valid],
            w_sample[valid],
            p0=[C, V],
            sigma=fit_sigmas[valid],
            absolute_sigma=True,
            bounds=([0.0, -np.inf], [np.inf, np.inf]),
            maxfev=5000,
        )
        bootstrap_c[idx], bootstrap_v[idx] = popt
    except (RuntimeError, ValueError, TypeError):
        pass

C_err = float(np.std(bootstrap_c[np.isfinite(bootstrap_c)])) if np.any(np.isfinite(bootstrap_c)) else None
V_err = float(np.std(bootstrap_v[np.isfinite(bootstrap_v)])) if np.any(np.isfinite(bootstrap_v)) else None

print("c_0:", f"{C:.8g}", "+/-", "n/a" if C_err is None else f"{C_err:.3g}")
print("V:", f"{V:.8g}", "+/-", "n/a" if V_err is None else f"{V_err:.3g}")
print("chi^2/dof:", "n/a" if chi2_dof is None else f"{chi2_dof:.6g}")

def _scatter_trace(x, y, yerr, *, name, color, symbol="circle"):
    yerr = np.asarray(yerr, dtype=float)
    visible_yerr = np.isfinite(yerr) & (yerr >= 0)
    plotted_yerr = np.where(visible_yerr, yerr, 0.0)
    return go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=name,
        marker={"size": MARKER_SIZE, "color": color, "symbol": symbol},
        error_y={"type": "data", "array": plotted_yerr, "visible": bool(np.any(visible_yerr)), "thickness": 1, "width": CAPSIZE},
        customdata=np.column_stack([yerr]),
        hovertemplate=(
            "T=%{x:g}<br>"
            "W=%{y:.8g}<br>"
            "bootstrap error=%{customdata[0]:.3g}"
            "<extra></extra>"
        ),
    )


def make_wilson_fit_pdf_figure(*, scale="linear", xlim=None, ylim=None):
    if scale not in {"linear", "log"}:
        raise ValueError(f"Unknown scale: {scale}")

    fig = go.Figure()
    excluded = ~fit_mask
    if np.any(excluded):
        fig.add_trace(
            _scatter_trace(
                T[excluded],
                W[excluded],
                W_err[excluded],
                name="outside fit range",
                color=EXCLUDED_COLOR,
                symbol="circle-open",
            )
        )

    fig.add_trace(
        _scatter_trace(
            T[fit_mask],
            W[fit_mask],
            W_err[fit_mask],
            name="data used in fit",
            color=POINT_COLOR,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t_line,
            y=w_fit,
            mode="lines",
            name="fit",
            line={"color": FIT_COLOR, "width": 2},
            hovertemplate="T=%{x:.3g}<br>fit=%{y:.8g}<extra></extra>",
        )
    )

    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title="T",
        yaxis_title="W(R,T)",
        margin=PDF_MARGIN,
        showlegend=True,
        legend={"x": 0.98, "xanchor": "right", "y": 0.98, "yanchor": "top"},
    )
    fig.update_yaxes(type=scale)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim))
    if ylim is not None:
        fig.update_yaxes(range=list(ylim))
    return fig


fig_linear = make_wilson_fit_pdf_figure(scale="linear", xlim=LINEAR_XLIM, ylim=LINEAR_YLIM)
fig_linear.write_image(str(PDF_PATH_LINEAR))
print(f"Saved PDF: {PDF_PATH_LINEAR}")

fig_log = make_wilson_fit_pdf_figure(scale="log")
fig_log.write_image(str(PDF_PATH_LOG))
print(f"Saved PDF: {PDF_PATH_LOG}")
fig_linear


c_0: 1.0508078 +/- 0.000376
V: 0.20419509 +/- 3.72e-05
chi^2/dof: 0.142193
Saved PDF: ../data/gradient_flow_wtemp_analysis/beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58/potential_t_over_a2_0p5/wilson_loop_fit_R4_linear.pdf
Saved PDF: ../data/gradient_flow_wtemp_analysis/beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58/potential_t_over_a2_0p5/wilson_loop_fit_R4_log.pdf
